# UNSW-NB15 Experiment 1 v2 — Superseded Branch Audit

**Purpose:** preserve the original source-imbalanced PRE-01/PRE-02/model branch as audit evidence without deleting, moving, or modifying it.

This notebook is housekeeping only. It does **not** alter the authoritative merged master or PRE-00. It writes a single supersession manifest under the dataset root and inventories affected old artifacts where present.


In [1]:
from pathlib import Path
import os, json, hashlib
from datetime import datetime, timezone

PROJECT_FOLDER_NAME='EXPERIMENT1_V2'
DATASET_FOLDER='UNSW-NB15'
PROJECT_ROOT_OVERRIDE=None

def find_project():
    if PROJECT_ROOT_OVERRIDE:
        p=Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not p.exists(): raise FileNotFoundError(p)
        return p
    candidates=[]
    for mount in [Path('/content/drive'), Path('/content/exp1v2_google_drive')]:
        for my in [mount/'MyDrive', mount/'My Drive']:
            if my.exists():
                direct=my/PROJECT_FOLDER_NAME
                if direct.exists(): candidates.append(direct.resolve())
    if len(dict.fromkeys(map(str,candidates)))==1:
        return Path(list(dict.fromkeys(map(str,candidates)))[0])
    try:
        from google.colab import drive
        alt=Path('/content/exp1v2_archive_drive')
        if not alt.exists() or not any(alt.iterdir()):
            drive.mount(str(alt), force_remount=False)
        for my in [alt/'MyDrive', alt/'My Drive']:
            if my.exists() and (my/PROJECT_FOLDER_NAME).exists(): return (my/PROJECT_FOLDER_NAME).resolve()
    except Exception:
        pass
    raise RuntimeError('Unable to resolve exactly one EXPERIMENT1_V2 project root. Set PROJECT_ROOT_OVERRIDE.')

PROJECT_ROOT=find_project()
DATASET_ROOT=PROJECT_ROOT/'DATASETS'/DATASET_FOLDER
if not DATASET_ROOT.exists(): raise FileNotFoundError(DATASET_ROOT)
print('PROJECT_ROOT:',PROJECT_ROOT)
print('DATASET_ROOT:',DATASET_ROOT)


Mounted at /content/exp1v2_archive_drive
PROJECT_ROOT: /content/exp1v2_archive_drive/MyDrive/EXPERIMENT1_V2
DATASET_ROOT: /content/exp1v2_archive_drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15


In [2]:
# Non-destructive inventory of the known superseded branch.
patterns=[
    'partitioning_s8_s11_v1_seed20260914',
    'preprocessing_representation_s12_s14_v2',
    'modelling/notebook01_ae_search/v1_0_0',
    'modelling/notebook02_dae_search/v1_0_0',
]
found=[]
for rel in patterns:
    p=DATASET_ROOT/rel
    if p.exists():
        files=sum(1 for x in p.rglob('*') if x.is_file())
        found.append({'relative_path':rel,'absolute_path':str(p),'file_count':files})

incident={
  'dataset':'UNSW-NB15',
  'protocol':'Experiment_1_v2',
  'status':'SUPERSEDED_FOR_SCIENTIFIC_INFERENCE',
  'created_at_utc':datetime.now(timezone.utc).isoformat(),
  'reason':'Original fingerprint-atomic PRE-01 produced severe source-capture imbalance across benign roles, making benign threshold calibration and development non-exchangeable.',
  'verified_diagnostic_findings':{
    'threshold_recomputed_exactly':True,
    'calibration_benign_exceedance_pct':1.0003,
    'prediction_rule_mismatches':0,
    'development_benign_median_reconstruction_error':8.5115,
    'calibration_benign_median_reconstruction_error':0.0213,
    'development_roc_auc_recomputed_approx':0.5025,
    'cause':'source-capture composition collapse under fingerprint-group allocation'
  },
  'retained_authoritative_upstream':['merged master','PRE-00 taxonomy','PRE-00 row identity','PRE-00 raw feature policy','PRE-00 feature fingerprints'],
  'invalidated_downstream':['original PRE-01 membership','original PRE-02 representation','Notebook 01 AE results','partial Notebook 02 DAE results','any later result derived from that upstream identity'],
  'archived_paths_detected':found,
  'deletion_performed':False,
}
out=DATASET_ROOT/'UNSW_PRE01_SOURCE_CAPTURE_INCIDENT_20260918.json'
if out.exists():
    old=json.loads(out.read_text())
    # Preserve the original incident record rather than overwrite it.
    print('Existing incident manifest preserved:',out)
else:
    out.write_text(json.dumps(incident,indent=2,sort_keys=True)+'\n',encoding='utf-8')
    print('Wrote incident manifest:',out)
print(json.dumps(incident,indent=2))


Wrote incident manifest: /content/exp1v2_archive_drive/MyDrive/EXPERIMENT1_V2/DATASETS/UNSW-NB15/UNSW_PRE01_SOURCE_CAPTURE_INCIDENT_20260918.json
{
  "dataset": "UNSW-NB15",
  "protocol": "Experiment_1_v2",
  "status": "SUPERSEDED_FOR_SCIENTIFIC_INFERENCE",
  "created_at_utc": "2026-09-18T14:32:59.847699+00:00",
  "reason": "Original fingerprint-atomic PRE-01 produced severe source-capture imbalance across benign roles, making benign threshold calibration and development non-exchangeable.",
  "verified_diagnostic_findings": {
    "threshold_recomputed_exactly": true,
    "calibration_benign_exceedance_pct": 1.0003,
    "prediction_rule_mismatches": 0,
    "development_benign_median_reconstruction_error": 8.5115,
    "calibration_benign_median_reconstruction_error": 0.0213,
    "development_roc_auc_recomputed_approx": 0.5025,
    "cause": "source-capture composition collapse under fingerprint-group allocation"
  },
  "retained_authoritative_upstream": [
    "merged master",
    "PRE-00 